# 04 - Curadoria de Stopwords por Document Frequency (DF)

Este notebook calcula a frequência documental dos termos no corpus pré-processado, identifica candidatos a ruído acima de um limiar configurável e exporta uma planilha em `data/processed/` para curadoria manual.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.curadoria import (
    calcular_document_frequency,
    carregar_stopwords_curadas,
    exportar_curadoria,
    filtrar_candidatos_ruido,
)

## 1. Parâmetros de Curadoria

In [ ]:
INPUT_FILE = PROJECT_ROOT / "data" / "processed" / "dataset_preprocessado.csv"
CURADORIA_FILE = PROJECT_ROOT / "data" / "processed" / "curadoria_stopwords.csv"

MIN_DF = 5
EDGE_REVIEW_PCT = 15.0  # Ponto de partida para triagem manual (em % de documentos)

## 2. Cálculo de Document Frequency (DF) e Exportação para Revisão

In [ ]:
df_corpus = pd.read_csv(INPUT_FILE, encoding="utf-8").dropna(subset=["clean_text"]).reset_index(drop=True)

table_df = calcular_document_frequency(df_corpus["clean_text"], min_df=MIN_DF)
display(table_df.head(50))

targets_review = filtrar_candidatos_ruido(table_df, edge_review_pct=EDGE_REVIEW_PCT)
exportar_curadoria(targets_review, CURADORIA_FILE)

## 3. Reconstrução das Stopwords Pós-Curadoria
Após preencher a coluna `decision` com `'excluir'` no arquivo CSV exportado, execute a célula abaixo para validar o conjunto final de stopwords.

In [ ]:
CUSTOM_STOPWORDS = carregar_stopwords_curadas(CURADORIA_FILE)
CUSTOM_STOPWORDS